# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. You will learn how to load the dataset, examine its schema, extract records, and perform basic exploratory data analysis (EDA). All entities (record sets, fields, columns) are referenced by their `@id` fields as required by Croissant best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The `metadata` field provides details (e.g., name and description) about the dataset
metadata_obj = dataset.metadata
print(f"{metadata_obj.name}: {metadata_obj.description}")


## 2. Data Overview
Review available record sets, their fields, and IDs. Record sets describe the main tables or entities within the dataset schema.

**Below, we list the record sets and, for each, their fields and columns, all by their `@id`.**

In [ ]:
# List all available record sets
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for recset in record_sets:
    print(f"RecordSet name: {recset.name}")
    print(f"  @id: {recset.id}")
    print(f"  Description: {getattr(recset, 'description', '')}")
    print("  Fields and Columns:")
    for field in recset.fields:
        print(f"    Field: {field.name}  (@id: {field.id})")
    for col in recset.columns:
        print(f"    Column: {col.name}  (@id: {col.id})")
    print("")


## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis.

The selected record set and field `@id`s are referenced by their Croissant `@id` values.


In [ ]:
# Select the main tabular record set (usually the one with the survey data)

# If you don't know the name, print record_set ids in previous cell
record_sets = dataset.record_sets
main_record_set = None
for recset in record_sets:
    # You can replace this logic or print all available options
    if "survey" in recset.name.lower() or "main" in recset.name.lower() or 'mental' in recset.name.lower():
        main_record_set = recset
        break
if not main_record_set:
    main_record_set = record_sets[0] # fallback to the first

main_record_set_id = main_record_set.id  # Use @id field
print(f"Using record set: {main_record_set.name} (@id={main_record_set_id})\n")

# You can extract all record set ids for batch processing
all_record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for rs_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set: {rs_id} with shape: {df.shape}")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# Inspect fields and head for the main table
print(f"\nMain DataFrame columns (@id):\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()


## 4. Exploratory Data Analysis (EDA)

Here we'll explore numerical fields in the dataset, apply simple filters, normalize a numeric score, and group by a demographic variable—referencing all variables by their `@id` as required.

**Example fields:**
- GAD-7 total score (e.g. `gad7_total_score`)
- PHQ-9 total score (e.g. `phq9_total_score`)
- Village or region field for grouping

**Find the actual `@id` from the overview above. Here we use sample ids you can replace as appropriate.**


In [ ]:
# --- Set @id values for demonstration. Replace with correct @id from previous cells! ---
# For example, the field representing GAD-7 total score may have @id like 'gad7_total_score'
# The field representing 'village' for demographic grouping could have @id like 'village'

numeric_field_id = None
group_field_id = None
# Try to find likely numeric and grouping columns
for col in dataframes[main_record_set_id].columns:
    if 'gad7' in col.lower() or 'phq9' in col.lower():
        numeric_field_id = col
    if 'village' in col.lower() or 'region' in col.lower():
        group_field_id = col

if numeric_field_id is None:
    numeric_field_id = dataframes[main_record_set_id].select_dtypes('number').columns[0]
    print(f"Defaulting numeric_field_id to: {numeric_field_id}")
if group_field_id is None:
    print("Could not automatically find a group field; please substitute with the correct `@id`. Proceeding without grouping...")

# Clean and drop NA for numeric analysis
df = dataframes[main_record_set_id].copy()
df = df.dropna(subset=[numeric_field_id])

threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")

# Normalize the numeric field (z-score)
filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Group by a demographic field, if chosen
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())


## 5. Visualization

Visualize data distributions and group differences using `matplotlib` or `seaborn`. We'll plot a histogram of GAD-7 (or PHQ-9) scores and (if grouping column is present) a bar chart by village.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion

This notebook provided an end-to-end workflow for exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`.

**Summary:**
- Loaded the schema and metadata using Croissant.
- Explored all record sets and their fields (referenced by `@id`).
- Loaded survey records into pandas DataFrames.
- Performed example EDA (numeric filtering, normalization, grouping).
- Visualized main assessment fields and demographic breakdowns.

Replace field and record set `@id`s as necessary for your specific analysis objectives. For further investigation, see the [mlcroissant documentation](https://mlcommons.org/croissant/) and the full dataset documentation.